In [ ]:
import random
import sys
import numpy as np
import torch

from functools import partial
from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"

DIRECTORY = INPUT_ROOT / COMPETITION
if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({path.parent.parent
                         for path in INPUT_ROOT.rglob("annotations_coco.json")
                         if path.parent.name == "train"})
    if len(data_roots) != 1:
        raise FileNotFoundError(
            "Attach the extracted competition data, or set DIRECTORY to its input folder."
        )
    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "data_cv.py",
    "dataset_helpers.py",
    "torch_model.py",
    "yolo26_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({path.parent for path in INPUT_ROOT.rglob("torch_model.py")
    if path.parent.name == "scripts" and all((path.parent / filename).is_file()
                                              for filename in required_scripts)
})
scripts_parent = str(script_dirs[0].parent)

if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations
from scripts.model_builders import build_ssd_model
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.experiment_runner import run_torch_final
from scripts.experiment_results import model_training_results_final

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
EPOCHS = 5
FRAME_STEP = 15
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
coco_train = COCO(str(INFO_TRAIN))
coco_val = COCO(str(INFO_VAL))

train_frames, train_boxes = load_annotations(coco_train, split="train")
valid_frames, valid_boxes = load_annotations(coco_val, split="val")

train_annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
valid_annotations_by_image = {image_id: group for image_id, group in valid_boxes.groupby("image_id")}
sampled_train_frames = (train_frames.sort_values(["video", "frame_id"]).groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

In [ ]:
train_dataset = DetectionDataset(frames=sampled_train_frames, directory=DIRECTORY,
                                 annotations_by_image=train_annotations_by_image,
                                 empty_annotations=train_boxes.iloc[:0].copy(), validation=False)
valid_dataset = DetectionDataset(frames=valid_frames, directory=DIRECTORY, 
                                 annotations_by_image=valid_annotations_by_image,
                                 empty_annotations=valid_boxes.iloc[:0].copy(), validation=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_data)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=collate_data)

In [ ]:
output_root = PROJECT_ROOT / "outputs" / "ssd_final"
train_kwargs = {
    "train_loader": train_loader,
    "valid_loader": valid_loader,
    "annotations_path": INFO_VAL,
    "device": device,
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "lr_scheduler": False,
    "step_size": 4
}

In [ ]:
final_history, final_results = run_torch_final(**train_kwargs, 
                                               model_method=partial(build_ssd_model, iou_threshold=0.4), 
                                               output_dir=output_root)

In [ ]:
final_results_wrap = model_training_results_final(output_root)